# Bronze Layer: Automated Ingestion & Extraction Notebook
Metadata-driven ETL pipeline that extracts source tables from MySQL via Bore TCP tunnel, handles both Full Load and Incremental Load (Spark SQL MERGE), tracks watermarks, and logs execution to the centralized metadata control table.

In [ ]:
# 1. Install Driver Libraries for Serverless / Community Compute
%pip install pymysql cryptography --quiet

In [ ]:
%run ../../src/utilities/logger

In [ ]:
%run ../../src/utilities/utils

In [ ]:
# 3. Initialize SparkSession and Task Logger
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp
import pymysql
import pandas as pd

# Ensure SparkSession is active
spark = SparkSession.builder.getOrCreate()

# Initialize logger for this notebook execution
logger = get_task_logger(notebook_name_override="bronze_extraction")
logger.info("=== Starting Bronze Extraction Pipeline Execution ===")


In [ ]:
# 4. Interactive Widgets for Dynamic Tunnel Port & Batch Filtering
dbutils.widgets.text("tunnel_port", "30577", "Bore Tunnel Port")
db_port = dbutils.widgets.get("tunnel_port")

dbutils.widgets.text("batch_id", "ALL", "Batch ID (1, 2, 3, 4 or ALL)")
batch_id = dbutils.widgets.get("batch_id").strip()

logger.info(f"Active Tunnel Port: {db_port} | Selected Batch ID: {batch_id}")
print(f"Active Tunnel Port: {db_port} | Selected Batch ID: {batch_id}")


In [ ]:
# 5. Retrieve Credentials Securely from Databricks Secrets
SECRET_SCOPE = "wanderbricks_scope"

db_user = dbutils.secrets.get(scope=SECRET_SCOPE, key="mysql_user")
db_password = dbutils.secrets.get(scope=SECRET_SCOPE, key="mysql_password")
db_host = dbutils.secrets.get(scope=SECRET_SCOPE, key="tunnel_host")
db_name = dbutils.secrets.get(scope=SECRET_SCOPE, key="mysql_db")

logger.info(f"Retrieved credentials from scope '{SECRET_SCOPE}' for user '{db_user}'")
print(f"Target database: {db_name} at host: {db_host}")

In [ ]:
# 6. Ensure Bronze Schema Exists in Unity Catalog
spark.sql("CREATE SCHEMA IF NOT EXISTS spark_training.bronze;")
logger.info("Ensured target schema 'spark_training.bronze' exists.")

In [ ]:
# 7. Metadata-Driven Ingestion Loop (Full Load + Incremental Load via spark.sql)
# Fetch active tables configured for the Bronze layer and selected batch
active_tables = get_active_tables(target_layer="bronze", batch_id=batch_id)

# Python-level defensive filter in case batch_id was not yet filtered in SQL
if batch_id and batch_id.upper() != "ALL":
    active_tables = [t for t in active_tables if str(t.get("batch_id", "")) == str(batch_id)]
    logger.info(f"Filtered for Batch ID '{batch_id}': Found {len(active_tables)} active tables to process.")
    print(f"Filtered for Batch ID '{batch_id}': Found {len(active_tables)} active Bronze tables to process.\n")
else:
    logger.info(f"Found {len(active_tables)} active tables for Bronze ingestion (All Batches).")
    print(f"Found {len(active_tables)} active Bronze tables to process (All Batches).\n")

if not active_tables:
    logger.warning(f"No active tables found matching Batch ID: '{batch_id}'. Skipping ingestion loop.")
    print(f"No active tables found matching Batch ID: '{batch_id}'.")

for metadata in active_tables:
    table_id = metadata["table_id"]
    source_table_name = metadata["source_table_name"]
    target_table = f"{metadata['target_database_name']}.{metadata['target_schema_name']}.{metadata['target_table_name']}"
    merge_key = metadata.get("merge_key")
    last_load_date_column = metadata.get("last_load_date_column")
    last_load_date = metadata.get("last_load_date")
    base_query = metadata["query"]
    load_type = (metadata.get("load_type") or "incremental").strip().lower()
    batch_num = metadata.get("batch_id", "N/A")

    logger.info(f"------------------------------------------------------------")
    logger.info(f"Processing Table ID {table_id} [Batch {batch_num}]: '{source_table_name}' -> '{target_table}' (Load Type: {load_type})")
    print(f"\nProcessing Table {table_id} [Batch {batch_num}]: {source_table_name} -> {target_table} ({load_type})")

    # 1. Build pushdown query (handling incremental watermark filtering)
    extract_query = base_query
    if "incremental" in load_type and last_load_date and last_load_date_column:
        if "where" in extract_query.lower():
            extract_query = f"{base_query} AND {last_load_date_column} > '{last_load_date}'"
        else:
            extract_query = f"{base_query} WHERE {last_load_date_column} > '{last_load_date}'"
        logger.info(f"Applying incremental watermark filter: {last_load_date_column} > '{last_load_date}'")

    # 2. Connect to MySQL and extract data
    connection = None
    try:
        update_last_load_status(table_id, "RUNNING")
        connection = pymysql.connect(
            host=db_host,
            port=int(db_port),
            user=db_user,
            password=db_password,
            database=db_name,
            connect_timeout=15
        )
        logger.info(f"Executing MySQL query: {extract_query}")
        pdf = pd.read_sql_query(extract_query, con=connection)
    except Exception as e:
        logger.error(f"Extraction failed for table {source_table_name}: {e}")
        update_last_load_status(table_id, "FAILED")
        raise e
    finally:
        if connection:
            connection.close()

    # 3. Check for extracted records
    if pdf.empty:
        logger.info(f"No new or modified records found for '{source_table_name}'. Skipping write.")
        update_last_load_status(table_id, "SUCCESS")
        print(f"  -> 0 new rows. Table is up to date.")
        continue

    # 4. In-memory count & PySpark DataFrame conversion with Arrow acceleration
    extracted_count = len(pdf)
    df = spark.createDataFrame(pdf)
    df = df.withColumn("created_on", current_timestamp()).withColumn("updated_on", current_timestamp())
    logger.info(f"Extracted {extracted_count} rows for '{source_table_name}'")
    print(f"  -> Extracted {extracted_count} rows.")

    # 5. Write to Delta Lake (Full Load vs Incremental Spark SQL MERGE)
    table_exists = spark.catalog.tableExists(target_table)
    try:
        if not table_exists or "full" in load_type:
            # Initial load or full refresh: Overwrite table
            df.write.format("delta").mode("overwrite").saveAsTable(target_table)
            logger.info(f"Saved {extracted_count} rows to '{target_table}' (mode: overwrite)")
            print(f"  -> Successfully written to {target_table} (overwrite)")
        else:
            # Incremental load: Perform Delta MERGE using spark.sql
            staging_view = f"staging_{source_table_name}"
            df.createOrReplaceTempView(staging_view)

            # Construct composite join condition (handles single or multiple PKs)
            keys = [k.strip() for k in merge_key.split(",")]
            join_condition = " AND ".join([f"t.{k} = s.{k}" for k in keys])

            # Base non-audit columns
            base_cols = [c for c in df.columns if c not in ["created_on", "updated_on"]]
            update_assignments = [f"t.{col} = s.{col}" for col in base_cols]
            update_assignments.append("t.updated_on = current_timestamp()")
            update_set_clause = ",\n                    ".join(update_assignments)

            all_cols = base_cols + ["created_on", "updated_on"]
            insert_cols_str = ", ".join(all_cols)
            insert_vals_str = ", ".join([f"s.{c}" for c in base_cols] + ["current_timestamp()", "current_timestamp()"])

            merge_query = f"""
                MERGE INTO {target_table} AS t
                USING {staging_view} AS s
                ON {join_condition}
                WHEN MATCHED THEN UPDATE SET
                    {update_set_clause}
                WHEN NOT MATCHED THEN INSERT ({insert_cols_str})
                    VALUES ({insert_vals_str})
            """

            logger.info(f"Executing Spark SQL MERGE for '{target_table}' on [{merge_key}]...")
            spark.sql(merge_query)
            logger.info(f"Successfully merged {extracted_count} rows into '{target_table}'")
            print(f"  -> Successfully merged {extracted_count} rows into {target_table} via spark.sql")

        # 6. Extract high watermark in-memory without extra Spark cluster jobs
        watermark_val = None
        if last_load_date_column and last_load_date_column in pdf.columns:
            max_val = pdf[last_load_date_column].max()
            if pd.notna(max_val):
                watermark_val = str(max_val)

        # 7. Atomically Update Watermark and Status in a single Delta transaction
        update_pipeline_metadata(table_id, status="SUCCESS", watermark=watermark_val)
        logger.info(f"Completed processing for table ID {table_id}: '{source_table_name}' successfully.")

    except Exception as e:
        logger.error(f"Delta write/merge failed for '{target_table}': {e}")
        update_last_load_status(table_id, "FAILED")
        raise e

logger.info("=== Finished Bronze Extraction Pipeline Execution for Selected Tables ===")
print("\nAll selected Bronze tables processed successfully!")
